In [ ]:
# ==========================================================
# RED NEURONAL PARA CLASIFICAR FLORES IRIS
# Dataset:
# https://www.kaggle.com/datasets/arshid/iris-flower-dataset

# ==========================================================
# 1. IMPORTAR LIBRERÍAS
# ==========================================================

import pandas as pd
import numpy as np

# Librerías de visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

# Subir archivos desde Google Colab
from google.colab import files
import io

# TensorFlow y Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input


In [ ]:
# ==========================================================
# 2. SUBIR EL DATASET DESDE TU EQUIPO
# ==========================================================

# Ejecuta esta celda y aparecera un boton para seleccionar
# el archivo iris.csv desde tu computadora.

uploaded = files.upload()

# El archivo subido se guarda en un diccionario {nombre: bytes}
nombre_archivo = list(uploaded.keys())[0]

# Leer el CSV desde los bytes recibidos
df = pd.read_csv(io.BytesIO(uploaded[nombre_archivo]))

# Mostrar las primeras filas
print("Primeros registros:")
print(df.head())

In [ ]:

# ==========================================================
# 3. PREPROCESAMIENTO DE DATOS
# ==========================================================

# Separar características (X) y etiquetas (y)
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values

# Convertir etiquetas de texto a números
# Ejemplo:
# Iris-setosa -> 0
# Iris-versicolor -> 1
# Iris-virginica -> 2

encoder = LabelEncoder()
y = encoder.fit_transform(y)

print("\nClases encontradas:")
print(encoder.classes_)

In [ ]:

# ==========================================================
# 4. DIVIDIR EL DATASET
# ==========================================================

# 80% entrenamiento
# 20% pruebas

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ==========================================================
# 5. NORMALIZAR LOS DATOS
# ==========================================================

# La normalización mejora el desempeño de la red neuronal.

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# ==========================================================
# 6. CONSTRUIR LA RED NEURONAL
# ==========================================================

# Arquitectura:
# Entrada: 4 neuronas (4 características)
# Capa oculta 1: 16 neuronas
# Capa oculta 2: 8 neuronas
# Salida: 3 neuronas (3 especies)

model = Sequential([

    Input(shape=(4,)),

    Dense(
        16,
        activation='relu'
    ),

    Dense(
        8,
        activation='relu'
    ),

    Dense(
        3,
        activation='softmax'
    )
])

# Mostrar resumen de la arquitectura
model.summary()

# ==========================================================
# 7. COMPILAR EL MODELO
# ==========================================================

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)



In [ ]:
# ==========================================================
# 8. ENTRENAR EL MODELO
# ==========================================================

print("\nEntrenando la red neuronal...\n")

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

# ==========================================================
# 9. EVALUAR EL MODELO
# ==========================================================

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\n================================")
print(f"Precisión del modelo: {accuracy*100:.2f}%")
print("================================")


In [ ]:
# ==========================================================
# 10. REALIZAR PREDICCIONES
# ==========================================================

predicciones = model.predict(X_test)

# Obtener la clase con mayor probabilidad
clases = np.argmax(predicciones, axis=1)

print("\nPrimeras 10 predicciones:")
for i in range(10):

    print(
        f"Predicción: {encoder.inverse_transform([clases[i]])[0]}"
        f" | Real: {encoder.inverse_transform([y_test[i]])[0]}"
    )

# ==========================================================
# 11. PROBAR UNA FLOR NUEVA
# ==========================================================

# Datos:
# SepalLength
# SepalWidth
# PetalLength
# PetalWidth

nueva_flor = np.array([
    [5.1, 3.5, 1.4, 0.2]
])

# Aplicar la misma normalización
nueva_flor = scaler.transform(nueva_flor)

# Realizar la predicción
resultado = model.predict(nueva_flor)

# Obtener la especie
especie = encoder.inverse_transform(
    [np.argmax(resultado)]
)

print("\nLa nueva flor pertenece a:")
print(especie[0])

# ==========================================================
# FIN DEL PROGRAMA
# ==========================================================

In [ ]:
# ==========================================================
# 12. VISUALIZAR RESULTADOS DEL ENTRENAMIENTO
# ==========================================================

# Gráfica de precisión (accuracy) y pérdida (loss)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Precisión
axes[0].plot(history.history['accuracy'], label='Entrenamiento')
axes[0].plot(history.history['val_accuracy'], label='Validación')
axes[0].set_title('Precisión durante el entrenamiento')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Precisión')
axes[0].legend(loc='lower right')
axes[0].grid(True)

# Pérdida
axes[1].plot(history.history['loss'], label='Entrenamiento')
axes[1].plot(history.history['val_loss'], label='Validación')
axes[1].set_title('Pérdida durante el entrenamiento')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Pérdida')
axes[1].legend(loc='upper right')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================================
# 13. MATRIZ DE CONFUSIÓN
# ==========================================================

# Predecir clases del conjunto de prueba
y_pred = np.argmax(model.predict(X_test), axis=1)

# Calcular matriz de confusión
matriz = confusion_matrix(y_test, y_pred)

# Visualizar
plt.figure(figsize=(6, 5))
sns.heatmap(
    matriz,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=encoder.classes_,
    yticklabels=encoder.classes_
)
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.tight_layout()
plt.show()